In [ ]:
# install reequired packages
%pip install pyserial
%pip install numpy pandas matplotlib
%pip install openpyxl

In [ ]:
#import packages that are used in this script
import numpy as np
import pandas as pd
import tkinter as tk
import serial
import time
import matplotlib.pyplot as plt
from IPython import display
from datetime import datetime
import os 
import sys
import re
import signal
import openpyxl
import glob

plt.style.use("dark_background")

In [ ]:
#Serial port magic

def serial_ports():
    """ Lists serial port names

        :raises EnvironmentError:
            On unsupported or unknown platforms
        :returns:
            A list of the serial ports available on the system
    """
    if sys.platform.startswith('win'):
        ports = ['COM%s' % (i + 1) for i in range(256)]
    elif sys.platform.startswith('linux') or sys.platform.startswith('cygwin'):
        # this excludes your current terminal "/dev/tty"
        ports = glob.glob('/dev/tty[A-Za-z]*')
    elif sys.platform.startswith('darwin'):
        ports = glob.glob('/dev/tty.*')
    else:
        raise EnvironmentError('Unsupported platform')

    result = []
    for port in ports:
        try:
            s = serial.Serial(port)
            s.close()
            result.append(port)
        except (OSError, serial.SerialException):
            pass
    return result

def findDevice(question="hello", answer="", flush=False, timeout=5):
    for port in serial_ports():
        with serial.Serial(port, baudrate=115200, timeout=timeout) as ser:
            time.sleep(2)  # let the board finish resetting after port open
            ser.reset_input_buffer()

            if flush:
                ser.flush()
                time.sleep(0.5)

            ser.write(question.encode())
            time.sleep(0.5)

            msg = ser.readline().decode(errors="replace")
            print(f"{port}: {msg!r}")
            if answer and answer in msg:
                print(f"Found device at: {port}, answer: {msg}")
                return port
    return None

def send_read_command(port, string, required_string=None, baudrate=115200, timeout=10):
    """
    Opens the serial port, sends a “hello” and then your command, and collects replies.
    Stops reading when either:
      • no more data arrives (an empty readline, i.e. per-read timeout fired), or
      • we see `required_string` in the response, if given, or
      • the overall `timeout` has elapsed.
    Returns the list of lines read so far.
    """
    lines = []
    start = time.monotonic()
    with serial.Serial(port, baudrate=baudrate, timeout=timeout) as ser:
        # hand-shake / mode set
        ser.setRTS(False)
        ser.flush()
        time.sleep(0.7)
        ser.write(b"hello\n")
        time.sleep(0.5)
        ser.write(f"{string}\n".encode())
        last_recv = time.monotonic()

        while True:
            # Check idle timeout
            now = time.monotonic()
            if now - last_recv >= timeout:
                print(f"No (additional) responses within user-set timeout: {timeout}s")
                return lines
            
            try:
                line = ser.readline().decode('utf-8', errors='replace').strip()
                # print(line)
                
                last_recv = time.monotonic() # Reset idle timer since we got data
            except Exception as e:
                print(f"Error reading line: {e!s}")
                break

            # if nothing arrived within per-read timeout, readline() returns b'' → line == ''
            if not line:
                print(f"No (additional) responses within user-set timeout: {timeout}s")
                return lines
                
            lines.append(line)

            # if we were waiting for a specific marker, stop when we see it
            if required_string and required_string in line:
                break
    return lines

In [ ]:
# The base eas
def measure_spec_raw(port=PORT, avg=15, integration_time=100000, gain=1, led = 0):
    with serial.Serial(port, baudrate=115200, timeout=20) as ser:
        time.sleep(2)
        ser.reset_input_buffer()

        if integration_time < 1280:
            integration_time = 1280

        if gain > 1:
            gain = 1

        if avg > 15:
            avg = 15 #avg over 15 will overload the arduino memory. Should also be capped at 15 in the arduino, but still.

        if led != 1:
            led = 0

        def send_command(cmd):
            ser.write((cmd + "\n").encode())
            return ser.readline().decode(errors="replace").strip()

        send_command(f"set_gain,{gain}")
        send_command(f"set_integration,{integration_time}")
        send_command(f"set_avg,{avg}")
        send_command(f"set_led,{led}")

        ser.write(b"spec,raw\n")
        spec_raw = ser.readline()
        spec_string = spec_raw.decode(errors="replace").strip()

        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        result = f"{timestamp}," + f"{avg}," + f"{integration_time}," + f"{gain}," + spec_string

        send_command(f"set_led,0")
        return result

# Set a dark spectrum, to be subtracted from a light spectrum later
dark_spectrum = None

def dark(port=PORT):
    global dark_spectrum

    with serial.Serial(port, baudrate=115200, timeout=20) as ser:
        time.sleep(2)
        ser.reset_input_buffer()

        def send_command(cmd):
            ser.write((cmd + "\n").encode())
            return ser.readline().decode(errors="replace").strip()

        send_command("set_gain,1")
        send_command("set_integration,100000")
        send_command("set_avg,10")
        send_command("set_led,0")

        ser.write(b"spec,raw\n")
        specdark = ser.readline()
        dark_spectrum = specdark.decode(errors="replace").strip()

    return dark_spectrum

dark()

def measure_spec(port=PORT, avg=15, integration_time=100000, gain=1, led=0):
    with serial.Serial(port, baudrate=115200, timeout=20) as ser:
        time.sleep(2)
        ser.reset_input_buffer()

        def send_command(cmd):
            ser.write((cmd + "\n").encode())
            return ser.readline().decode(errors="replace").strip()

        send_command(f"set_gain,{gain}")
        send_command(f"set_integration,{integration_time}")
        send_command(f"set_avg,{avg}")
        send_command(f"set_led,{led}")

        ser.write(b"spec,raw\n")
        spec_raw1 = ser.readline()
        spec_light1 = spec_raw1.decode(errors="replace").strip()

        send_command(f"set_led,0")

    spec_light = np.array(spec_light1.split(","), dtype=int)

    if dark_spectrum is None:
        raise ValueError("No dark spectrum has been measured. Call dark() first.")

    spec_dark = np.array(dark_spectrum.split(","), dtype=int)

    spec = np.maximum(spec_light - spec_dark, 0)

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    spec_str = ",".join(spec.astype(str))

    result = f"{timestamp},{avg},{integration_time},{gain},{spec_str}"

    return result

## Measurement 1: Taking raw spectra and plotting them ##

In [ ]:
# Take multiple raw spectra and plot them in one graph.
#specify all configs for the plot [avg (max 63), integration time (µs, min 1280), gain (0 or 1, low or high), LED on/off (1 or 0)].

configs = [
    [10, 50000, 1, 0],
    [10, 100000, 1, 0],
    [10, 200000, 1, 0],
]

n_pixels = 256
columns = ["timestamp", "avg", "integration_time", "gain"] + [f"pixel{i}" for i in range(n_pixels)]

results = []

for config in configs:
    print(f"Measuring with config: {config}")
    read = measure_spec_raw(PORT, *config).split(",")
    results.append(read)

data = pd.DataFrame(results, columns=columns)

# Numeric columns come in as strings from the split - convert them.
numeric_cols = ["avg", "integration_time", "gain"] + [f"pixel{i}" for i in range(n_pixels)]
data[numeric_cols] = data[numeric_cols].astype(int)

first_timestamp = data.loc[0, "timestamp"]
safe_timestamp = first_timestamp.replace(":", "-").replace(" ", "_")
data.to_csv(os.path.join("data", f"spec {safe_timestamp}_data.csv"), index=False)

# --- Plotting: use ONLY the pixel columns ---
pixel_cols = [c for c in data.columns if c.startswith("pixel")]
x = np.linspace(340, 780, len(pixel_cols))

for i in range(len(data)):
    y = data.loc[i, pixel_cols]
    avg, int_time, gain = data.loc[i, 'avg'], data.loc[i, 'integration_time'], data.loc[i, 'gain']
    label = f"gain={gain}, int_time={int_time}µs, avg={avg}"
    plt.plot(x, y, label=label)

plt.legend()
plt.text(-0.1, 1.15, f"{safe_timestamp}", size=8, ha="left", va="top", transform=plt.gca().transAxes)
plt.ylim(0, 4095)  # we have a signed 13-bit ADC. Voltage will not be negative, so functionally 12 bits
plt.xlabel("Wavelength (nm)")
plt.ylabel("Raw ADC counts")
plt.title("\n" + "Spectrometer Readings\n")
plt.show()

## Meassurement 2: Taking two spectra and subtracting them from eachother. ##

In [ ]:
# Subtraction Spectra
# This code is similor to the one used above, but can be used to take two spectra and subtract the first from the second spectrum. 
# In the current case, 

#specify all configs for the plot [avg (max 63), integration time (µs, min 1280), gain (0 or 1, low or high), LED (0 or 1, On or off)]
configs = [
    [10, 100000, 1, 0],
    [10, 100000, 1, 1],
]

n_pixels = 256
columns = ["timestamp", "avg", "integration_time", "gain"] + [f"pixel{i}" for i in range(n_pixels)]

results = []

for config in configs:
    spectrum = measure_spec(
        avg=config[0],
        integration_time=config[1],
        gain=config[2],
        led=config[3]
    )
    results.append(spectrum.split(","))

data = pd.DataFrame(results, columns=columns)

# Numeric columns come in as strings from the split - convert them.
numeric_cols = ["avg", "integration_time", "gain"] + [f"pixel{i}" for i in range(n_pixels)]
data[numeric_cols] = data[numeric_cols].astype(int)

first_timestamp = data.loc[0, "timestamp"]
safe_timestamp = first_timestamp.replace(":", "-").replace(" ", "_")
data.to_csv(os.path.join("data", f"spec {safe_timestamp}_data.csv"), index=False)

# --- Plotting: use ONLY the pixel columns ---
pixel_cols = [c for c in data.columns if c.startswith("pixel")]
x = np.linspace(340, 780, len(pixel_cols))

Spec_subtraction = np.subtract(data.loc[1, pixel_cols], data.loc[0, pixel_cols])

y = Spec_subtraction
avg, int_time, gain = data.loc[1, 'avg'], data.loc[1, 'integration_time'], data.loc[1, 'gain']
label = f"gain={gain}, int_time={int_time}µs, avg={avg}"
plt.plot(x, y, label=label)

plt.legend()
plt.text(-0.1, 1.15, f"{safe_timestamp}", size=8, ha="left", va="top", transform=plt.gca().transAxes)
plt.ylim(0, 4096)  # we have a 10-bit ADC counts
plt.xlabel("Wavelength (nm)")
plt.ylabel("Raw ADC counts")
plt.title("\n" + "Spectrometer Readings\n")
plt.show()